### Import libraries

In [52]:
import os, json, glob, gc, time
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import collections.abc

from darts import TimeSeries
from darts.dataprocessing.transformers import StaticCovariatesTransformer
from darts.models import TFTModel
from pytorch_lightning.callbacks import EarlyStopping


### Config

In [53]:
local_test_dir = r"C:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Model_for_scooters_only\local_test_scooter_data"   # use absolute path if notebooks differ in cwd
local_train_dir  = r"C:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Model_for_scooters_only\local_train_scooter_data"
group_keys_path = r"C:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Model_for_scooters_only\saved_group_keys_scooter\group_keys_scooter.json"

CACHE_DIR       = os.path.join(os.getcwd(),'series_cache_scooter')          # per-series .npz files live here

time_col   = 'CAL_DATE'
group_col  = 'PARENT_DEALER_CODE_MODEL_FAMILY'
target_col = 'NET_SALES'
FREQ       = 'D'

TRAIN_END = pd.Timestamp("2026-04-30")
VAL_START = pd.Timestamp("2026-05-01")
VAL_END   = pd.Timestamp("2026-07-30")

INPUT_CHUNK_LENGTH  = 365
OUTPUT_CHUNK_LENGTH = 154
TEST_HORIZON        = 154

static_covariates = [
    'PARENT_DEALER_CODE', 'MODEL_FAMILY', 'MODEL_NAME', 'BRAKE_TYPE',
    'IGNITION_TYPE', 'WHEEL_TYPE', 'COLOUR', 'DEALER_CITY',
    'X_CITY_CATEGORY', 'ZONAL_OFFICE_NAME'
]

future_covariates = [
    'NEW_YEAR','LOHRI','MAKAR_SANKRANTI','REPUBLIC_DAY','VASANT_PANCHAMI',
    'MAHA_SHIVRATRI','EID_UL_FITR','HOLIKA_DAHAN','HOLI','HANUMAN_JAYANTI',
    'AKSHAYA_TRITYA','BUDDHA_PURNIMA','GANGA_DUSSEHRA','JAGANNATH_RATHYATRA',
    'GURU_PURNIMA','NAG_PANCHAMI','RAKSHA_BANDHAN','HARTALIK_TEEJ',
    'GANESH_CHATURTHI','JANMASHTAMI','VISHWAKARMA_PUJA','KARWA_CHAUTH',
    'ONAM','MARRIAGE_DAY',
    'N-16','N-15','N-14','N-13','N-12','N-11','N-10','N-9','N-8','N-7',
    'N-6','N-5','N-4','N-3','N-2','N-1','N','N+1','N+2','N+3','N+4',
    'N+5','N+6','N+7','N+8','N+9','N+10',
    'D-3','D-2','D-1','D','D+1','D+2','D+3','D+4','D+5','D+6',
    'C','C+1','C+2','C+3','C+4','C+5','C+6'
]

penalty_cols = [
    'N-16','N-15','N-14','N-13','N-12','N-11','N-10','N-9','N-8','N-7',
    'N-6','N-5','N-4','N-3','N-2','N-1','N','N+1','N+2','N+3','N+4',
    'N+5','N+6','N+7','N+8','N+9','N+10',
    'D-3','D-2','D-1','D','D+1','D+2','D+3','D+4','D+5','D+6',
    'C','C+1','C+2','C+3','C+4','C+5','C+6'
]

val_window_days = (VAL_END - VAL_START).days + 1                       # 181
warmup_days     = INPUT_CHUNK_LENGTH + OUTPUT_CHUNK_LENGTH - val_window_days  # 368
warmup_start    = VAL_START - pd.Timedelta(days=warmup_days)
MIN_LEN         = INPUT_CHUNK_LENGTH + OUTPUT_CHUNK_LENGTH             # 549


### Custom loss function

In [54]:
class HuberMaeFeatureLoss(nn.HuberLoss):
    """
    Huber on component 0, with two festive adjustments driven by component 1
    of the target (the 0/1 festive flag):
      1. festive days are weighted `festive_weight` times more heavily
      2. UNDER-prediction on festive days carries an extra one-sided penalty

    The one-sidedness is the point. The previous version added a symmetric
    `flag * |y - y_hat|`, which penalised over- and under-shoot equally and so
    pulled festive predictions toward the conditional MEDIAN -- downward on a
    right-skewed festive distribution, i.e. the opposite of the intent.
    """

    def __init__(self, delta=1.0, festive_weight=3.0, under_weight=3.0, reduction='mean'):
        super().__init__(reduction='none', delta=delta)
        self.festive_weight = festive_weight
        self.under_weight = under_weight
        self.user_reduction = reduction

    def forward(self, input: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        y_hat = input[..., 0]
        y     = target[..., 0]
        flag  = target[..., 1]
        
        if not hasattr(self, "_dbg"):
            print("flag uniques:", torch.unique(flag)[:8])
            print(f"y range: min {float(y.min()):.2f} max {float(y.max()):.2f} mean {float(y.mean()):.2f}")
            self._dbg = True

        base       = super().forward(y_hat, y)
        weighted   = (1.0 + self.festive_weight * flag) * base
        undershoot = torch.clamp(y - y_hat, min=0)          # >0 only when too low
        total      = weighted + self.under_weight * flag * undershoot

        if self.user_reduction == 'mean':
            return total.mean()

        if self.user_reduction == 'sum':
            return total.sum()

        return total

In [55]:

print("="*60)
print("SECTION 2: BUILDING PER-SERIES CACHE")
print("="*60)

os.makedirs(CACHE_DIR, exist_ok=True)
manifest_path = os.path.join(CACHE_DIR, "manifest.json")

def safe_name(key):
    return str(key).replace("<>", "_").replace("/", "_").replace("\\", "_")

if os.path.exists(manifest_path):
    print("Cache already exists — loading manifest.")
    with open(manifest_path, "r") as f:
        manifest = json.load(f)
else:
    needed_cols = [time_col, group_col, target_col] + static_covariates + penalty_cols
    chunk_files = sorted(glob.glob(os.path.join(local_train_dir, "chunk_*.parquet")))
    print(f"Scanning {len(chunk_files)} chunk files...")

    series_keys  = []
    has_val      = []
    scaler_stats = {}
    static_rows  = []

    for ci, chunk_path in enumerate(chunk_files):
        df = pd.read_parquet(chunk_path, columns=needed_cols)
        df[time_col] = pd.to_datetime(df[time_col])

        for key, g in df.groupby(group_col, sort=False):
            g = g.sort_values(time_col).reset_index(drop=True)
            flag = (g[penalty_cols] != 0).any(axis=1).to_numpy(dtype=np.float32)

            t = g[time_col]
            tr = (t <= TRAIN_END).to_numpy()
            va = ((t >= warmup_start) & (t <= VAL_END)).to_numpy()

            if tr.sum() < MIN_LEN:
                continue   # too short to yield even one training sample

            sales = g[target_col].to_numpy(dtype=np.float32)
            tr_sales, tr_flag = sales[tr], flag[tr]

            # scaling params from TRAIN window only (no leakage from val)
            lo = 0.0
            hi = float(tr_sales.mean())
            if hi < 1e-3:
                hi = 1.0            # constant series guard

            keep_val = va.sum() >= MIN_LEN
            payload = {
                "train_sales": tr_sales,
                "train_flag":  tr_flag,
                "train_start": np.array(str(t[tr].iloc[0].date())),
            }
            if keep_val:
                payload["val_sales"] = sales[va]
                payload["val_flag"]  = flag[va]
                payload["val_start"] = np.array(str(t[va].iloc[0].date()))

            np.savez(os.path.join(CACHE_DIR, f"{safe_name(key)}.npz"), **payload)

            series_keys.append(str(key))
            has_val.append(bool(keep_val))
            scaler_stats[str(key)] = [lo, hi]
            static_rows.append(g[static_covariates].iloc[0].to_dict())

        del df
        gc.collect()
        print(f"  chunk {ci+1}/{len(chunk_files)} done — series so far: {len(series_keys)}")

    pd.DataFrame(static_rows).to_parquet(
        os.path.join(CACHE_DIR, "static_covariates.parquet"), index=False
    )
    manifest = {"series_keys": series_keys, "has_val": has_val,
                "scaler_stats": scaler_stats}
    with open(manifest_path, "w") as f:
        json.dump(manifest, f)
    del static_rows
    gc.collect()

series_keys  = manifest["series_keys"]
has_val      = manifest["has_val"]
scaler_stats = manifest["scaler_stats"]
print(f"\nSeries cached          : {len(series_keys)}")
print(f"Series with val strip  : {sum(has_val)}")


SECTION 2: BUILDING PER-SERIES CACHE
Cache already exists — loading manifest.

Series cached          : 34633
Series with val strip  : 34633


In [56]:
import numpy as np, os, json

new_stats = {}
for key in manifest["series_keys"]:
    with np.load(os.path.join(CACHE_DIR, f"{safe_name(key)}.npz")) as z:
        s = z["train_sales"]
    m = float(s.mean())
    new_stats[key] = [0.0, max(m, 0.05)]      # was: m if m > 1e-3 else 1.0

manifest["scaler_stats"] = new_stats
with open(os.path.join(CACHE_DIR, "manifest.json"), "w") as f:
    json.dump(manifest, f)
scaler_stats = new_stats
print("scaler_stats recomputed (guard 0.05)")

scaler_stats recomputed (guard 0.05)


In [57]:
print("\n" + "="*60)
print("SECTION 3: STATIC COVARIATES")
print("="*60)

from sklearn.preprocessing import OrdinalEncoder

static_df_all = pd.read_parquet(os.path.join(CACHE_DIR, "static_covariates.parquet"))

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
encoded_arr = encoder.fit_transform(
    static_df_all[static_covariates].astype(str)
).astype(np.float32)

encoded_df = pd.DataFrame(encoded_arr, columns=static_covariates)
STATIC_ENCODED = [
    encoded_df.iloc[[i]].reset_index(drop=True)
    for i in range(len(encoded_df))
]

print(f"Encoded static covariates for {len(STATIC_ENCODED)} series.")
for c in static_covariates:
    print(f"  {c}: {static_df_all[c].nunique()} categories")



SECTION 3: STATIC COVARIATES
Encoded static covariates for 34633 series.
  PARENT_DEALER_CODE: 1164 categories
  MODEL_FAMILY: 3 categories
  MODEL_NAME: 3 categories
  BRAKE_TYPE: 2 categories
  IGNITION_TYPE: 1 categories
  WHEEL_TYPE: 2 categories
  COLOUR: 13 categories
  DEALER_CITY: 852 categories
  X_CITY_CATEGORY: 3 categories
  ZONAL_OFFICE_NAME: 5 categories


In [58]:
print("\n" + "="*60)
print("SECTION 4: SHARED FUTURE COVARIATES")
print("="*60)

cov_cols = [time_col] + future_covariates
train_chunk0 = sorted(glob.glob(os.path.join(local_train_dir, "chunk_*.parquet")))[0]
test_chunk0  = sorted(glob.glob(os.path.join(local_test_dir,  "chunk_*.parquet")))[0]

cal = pd.concat([
    pd.read_parquet(train_chunk0, columns=cov_cols),
    pd.read_parquet(test_chunk0,  columns=cov_cols),
])
cal[time_col] = pd.to_datetime(cal[time_col])
cal = cal.drop_duplicates(subset=time_col).sort_values(time_col).reset_index(drop=True)

SHARED_COV = TimeSeries.from_dataframe(
    cal, time_col=time_col, value_cols=future_covariates,
    freq=FREQ, fill_missing_dates=False
).astype(np.float32)

print(f"Covariate calendar: {cal[time_col].min().date()} → {cal[time_col].max().date()} "
      f"({len(cal)} days, {len(future_covariates)} cols)")
del cal
gc.collect()



SECTION 4: SHARED FUTURE COVARIATES
Covariate calendar: 2023-04-01 → 2026-12-31 (1371 days, 68 cols)


0

In [59]:
# ---- drop series that never sold during training ----
DROP_NEVER_SOLD  = True
DORMANT_DAYS     = None      # set to 180/270/365 once the backtest tells you which

keep = []
for k in series_keys:
    with np.load(os.path.join(CACHE_DIR, f"{safe_name(k)}.npz")) as z:
        s = z["train_sales"]
    if DROP_NEVER_SOLD and s.sum() == 0:
        keep.append(False); continue
    if DORMANT_DAYS and s[-DORMANT_DAYS:].sum() == 0:
        keep.append(False); continue
    keep.append(True)

n_before = len(series_keys)
series_keys    = [k for k, m in zip(series_keys, keep) if m]
has_val        = [h for h, m in zip(has_val, keep) if m]
STATIC_ENCODED = [s for s, m in zip(STATIC_ENCODED, keep) if m]

assert len(series_keys) == len(has_val) == len(STATIC_ENCODED)
print(f"series: {n_before:,} -> {len(series_keys):,}  (dropped {n_before-len(series_keys):,})")


series: 34,633 -> 32,568  (dropped 2,065)


In [60]:

class DiskLazyTargetSequence(collections.abc.Sequence):
    """
    Target series backed by per-series .npz files.

    Returns a 2-component TimeSeries: [scaled NET_SALES, raw FESTIVE_FLAG].
    Scaling is applied at read time using train-window min/max.

    With cache_in_ram=True, arrays are held after first read. The cache is
    populated after worker fork, so it is never pickled — worker spawn stays
    cheap while per-epoch disk I/O disappears after epoch 1.
    """

    def __init__(self, cache_dir, series_keys, scaler_stats, static_encoded,
                 split="train", freq='D', cache_in_ram=True):
        self.cache_dir      = cache_dir
        self.series_keys    = series_keys
        self.scaler_stats   = scaler_stats
        self.static_encoded = static_encoded
        self.split          = split
        self.freq           = freq
        self.cache_in_ram   = cache_in_ram
        self._ram           = {} if cache_in_ram else None

    def __len__(self):
        return len(self.series_keys)

    # keep the RAM cache out of anything pickled to workers
    def __getstate__(self):
        state = self.__dict__.copy()
        state["_ram"] = {} if self.cache_in_ram else None
        return state

    def __getitem__(self, idx):
        if isinstance(idx, slice):
            return [self[i] for i in range(*idx.indices(len(self)))]
        if idx < 0:
            idx += len(self)
        if not 0 <= idx < len(self):
            raise IndexError(idx)

        # cache the FINISHED TimeSeries — after first touch this is a dict lookup
        if self._ram is not None and idx in self._ram:
            return self._ram[idx]

        key  = self.series_keys[idx]
        path = os.path.join(self.cache_dir, f"{safe_name(key)}.npz")
        with np.load(path, allow_pickle=False) as z:
            sales = z[f"{self.split}_sales"]
            flag  = z[f"{self.split}_flag"]
            start = str(z[f"{self.split}_start"])

        lo, hi = self.scaler_stats[key]
        scaled = ((sales - lo) / (hi - lo)).astype(np.float32)
        values = np.stack([scaled, flag], axis=1)
        times  = pd.date_range(start=start, periods=len(values), freq=self.freq)

        ts = TimeSeries.from_times_and_values(
            times, values,
            columns=[target_col, "FESTIVE_FLAG"],
            static_covariates=self.static_encoded[idx],
        )
        if self._ram is not None:
            self._ram[idx] = ts
        return ts

class SharedCovSequence(collections.abc.Sequence):
    """Returns the same in-RAM covariate TimeSeries for every index."""

    def __init__(self, shared_series, n):
        self.shared = shared_series
        self.n = n

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        if isinstance(idx, slice):
            return [self.shared for _ in range(*idx.indices(self.n))]
        if idx < 0:
            idx += self.n
        if not 0 <= idx < self.n:
            raise IndexError(idx)
        return self.shared


# --- train sequences (all series) ---
train_seq = DiskLazyTargetSequence(
    CACHE_DIR, series_keys, scaler_stats, STATIC_ENCODED, split="train", freq=FREQ
)
train_cov_seq = SharedCovSequence(SHARED_COV, len(series_keys))

# --- val sequences (only series that have a val strip) ---
val_keys    = [k for k, h in zip(series_keys, has_val) if h]
val_statics = [s for s, h in zip(STATIC_ENCODED, has_val) if h]
val_seq = DiskLazyTargetSequence(
    CACHE_DIR, val_keys, scaler_stats, val_statics, split="val", freq=FREQ
)
val_cov_seq = SharedCovSequence(SHARED_COV, len(val_keys))

print("\n" + "="*60)
print("SECTION 5: SANITY CHECK")
print("="*60)
_t0 = time.time()
s0 = train_seq[0]
print(f"train_seq[0]: {s0.start_time().date()} → {s0.end_time().date()} | "
      f"len={len(s0)} | comps={s0.components.tolist()}")
print(f"single lazy read took {(time.time()-_t0)*1000:.1f} ms")
print(f"train series: {len(train_seq)} | val series: {len(val_seq)}")

# Read-speed probe: this number decides whether disk-lazy is viable at all
_t0 = time.time()
for i in np.random.randint(0, len(train_seq), 200):
    _ = train_seq[int(i)]
_per_read = (time.time() - _t0) / 200
print(f"\nAvg random read: {_per_read*1000:.2f} ms")
print(f"→ est. time for 1 epoch at 50 samples/series: "
      f"{_per_read * len(train_seq) * 50 / 3600:.1f} h of pure disk I/O")


SECTION 5: SANITY CHECK
train_seq[0]: 2023-04-01 → 2026-04-30 | len=1126 | comps=['NET_SALES', 'FESTIVE_FLAG']
single lazy read took 2.0 ms
train series: 32568 | val series: 32568

Avg random read: 1.29 ms
→ est. time for 1 epoch at 50 samples/series: 0.6 h of pure disk I/O


In [61]:
import time, sys
t0 = time.time()
for i in range(len(train_seq)):
    _ = train_seq[i]
    if i % 5000 == 0:
        print(f"  train {i}/{len(train_seq)}  {time.time()-t0:.0f}s", flush=True)
for i in range(len(val_seq)):
    _ = val_seq[i]
print(f"pre-warmed {len(train_seq)+len(val_seq)} series in {time.time()-t0:.0f}s")

import psutil, os
print(f"RSS now: {psutil.Process(os.getpid()).memory_info().rss/1e9:.1f} GB")

  train 0/32568  0s
  train 5000/32568  7s
  train 10000/32568  13s
  train 15000/32568  20s
  train 20000/32568  26s
  train 25000/32568  33s
  train 30000/32568  42s
pre-warmed 65136 series in 90s
RSS now: 5.9 GB


In [62]:
print("\n" + "="*60)
print("SECTION 7: MODEL")
print("="*60)

torch.set_float32_matmul_precision('high')

now = datetime.now().strftime("%Y-%m-%d_%H_%M_%S")
MODEL_NAME = f"daily_tft_festive_lazy__scooters_{now}"
print("Model name:", MODEL_NAME)

early_stopping = EarlyStopping(
    monitor="val_loss", patience=5, min_delta=1e-4, mode="min"
)

model = TFTModel(
    input_chunk_length=INPUT_CHUNK_LENGTH,
    output_chunk_length=OUTPUT_CHUNK_LENGTH,

    hidden_size=32,
    lstm_layers=4,
    num_attention_heads=4,          # was 16 -> 4x less attention memory, and
                                    # 32/4 = 8 dims per head instead of 2
    dropout=0.05,

    batch_size=64,                  # was 256 -> 4x less memory per step
    n_epochs=100,
    use_reversible_instance_norm = True,
    likelihood=None,
    loss_fn=HuberMaeFeatureLoss(delta=100.0, festive_weight=3.0, under_weight=3.0),

    random_state=42,
    add_relative_index=True,

    save_checkpoints=True,
    force_reset=True,
    model_name=MODEL_NAME,
    skip_interpolation=True,


    pl_trainer_kwargs={
        "accelerator": "gpu" if torch.cuda.is_available() else "cpu",
        "devices": 1,
        "callbacks": [early_stopping],
        "gradient_clip_val": 0.1,
        "precision": "bf16-mixed",
        "accumulate_grad_batches": 4,
        "limit_train_batches": 5000,
        "limit_val_batches": 500,
    },

)


SECTION 7: MODEL
Model name: daily_tft_festive_lazy__scooters_2026-08-02_16_06_15


### RAM logging

In [63]:
import psutil, threading, time, os

_proc = psutil.Process(os.getpid())
_stop = threading.Event()

def _log_ram(interval=300):
    peak = 0
    while not _stop.is_set():
        rss  = _proc.memory_info().rss / 1e9
        kids = sum(c.memory_info().rss for c in _proc.children(recursive=True)) / 1e9
        vm   = psutil.virtual_memory()
        peak = max(peak, rss + kids)
        print(f"[{time.strftime('%H:%M:%S')}] main: {rss:.1f} GB | "
              f"workers: {kids:.1f} GB | total: {rss+kids:.1f} GB (peak {peak:.1f}) | "
              f"system: {vm.used/1e9:.1f}/{vm.total/1e9:.1f} GB", flush=True)
        _stop.wait(interval)

_stop.clear()
threading.Thread(target=_log_ram, daemon=True).start()
print("RAM logger started.")


[16:06:15] main: 5.9 GB | workers: 0.0 GB | total: 5.9 GB (peak 5.9) | system: 35.0/67.9 GB
RAM logger started.


In [64]:
import gc, torch

# free anything left on the card by a previous failed fit
for name in ("trainer", "model"):
    pass  # keep `model` — only clear stale CUDA allocations
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

free, total = torch.cuda.mem_get_info()
print(f"GPU free: {free/1e9:.2f} GB / {total/1e9:.2f} GB")
if free / total < 0.8:
    print("WARNING: less than 80% of the card is free -- restart the kernel before training.")

GPU free: 15.94 GB / 17.18 GB


In [65]:
print("\n" + "="*60)
print("SECTION 8: TRAINING")
print("="*60)

try:
    model.fit(
    series=train_seq,
    future_covariates=train_cov_seq,
    val_series=val_seq,
    val_future_covariates=val_cov_seq,
    max_samples_per_ts=400,          # REINSTATED
    dataloader_kwargs={
        "num_workers": 0,
        "pin_memory": True,
    },
    verbose=True,
)

finally:
    _stop.set()
    print("RAM logger stopped.")


SECTION 8: TRAINING


Detected user-defined float16-like precision. For mixed precision training, recommended options are 'bf16-mixed' and '16-mixed'.
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                              | Type                             | Params | Mode 
------------------------------------------------------------------------------------------------
0  | criterion                         | HuberMaeFeatureLoss              | 0      | train
1  | train_criterion                   | HuberMaeFeatureLoss              | 0      | train
2  | val_criterion                     | HuberMaeFeatureLoss              | 0      | train
3  | train_metrics                     | MetricCollection                 | 0      | train
4  | val_metrics                       | MetricCollection                 | 0      | train
5  | rin              

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

flag uniques: tensor([0.], device='cuda:0')
y range: min 0.00 max 0.00 mean 0.00


Training: |          | 0/? [00:00<?, ?it/s]

...

[18:36:16] main: 4.7 GB | workers: 0.0 GB | total: 4.7 GB (peak 5.9) | system: 31.4/67.9 GB
[18:41:16] main: 4.7 GB | workers: 0.0 GB | total: 4.7 GB (peak 5.9) | system: 31.5/67.9 GB
[18:46:16] main: 4.7 GB | workers: 0.0 GB | total: 4.7 GB (peak 5.9) | system: 31.7/67.9 GB
[18:51:16] main: 4.7 GB | workers: 0.0 GB | total: 4.7 GB (peak 5.9) | system: 31.7/67.9 GB
[18:56:16] main: 4.7 GB | workers: 0.0 GB | total: 4.7 GB (peak 5.9) | system: 31.7/67.9 GB
[19:01:16] main: 4.7 GB | workers: 0.0 GB | total: 4.7 GB (peak 5.9) | system: 31.7/67.9 GB
[19:06:16] main: 4.7 GB | workers: 0.0 GB | total: 4.7 GB (peak 5.9) | system: 31.8/67.9 GB


Validation: |          | 0/? [00:00<?, ?it/s]

[19:11:16] main: 4.7 GB | workers: 0.0 GB | total: 4.7 GB (peak 5.9) | system: 31.8/67.9 GB
[19:16:16] main: 4.8 GB | workers: 0.0 GB | total: 4.8 GB (peak 5.9) | system: 32.0/67.9 GB
[19:21:16] main: 4.8 GB | workers: 0.0 GB | total: 4.8 GB (peak 5.9) | system: 32.0/67.9 GB
[19:26:16] main: 4.8 GB | workers: 0.0 GB | total: 4.8 GB (peak 5.9) | system: 32.1/67.9 GB
[19:31:16] main: 4.8 GB | workers: 0.0 GB | total: 4.8 GB (peak 5.9) | system: 31.7/67.9 GB
[19:36:17] main: 4.8 GB | workers: 0.0 GB | total: 4.8 GB (peak 5.9) | system: 31.7/67.9 GB
[19:41:17] main: 4.8 GB | workers: 0.0 GB | total: 4.8 GB (peak 5.9) | system: 31.7/67.9 GB
[19:46:17] main: 4.8 GB | workers: 0.0 GB | total: 4.8 GB (peak 5.9) | system: 31.9/67.9 GB


Validation: |          | 0/? [00:00<?, ?it/s]

RAM logger stopped.


### Prediction

In [66]:
SHARED_COV.to_pickle(os.path.join(CACHE_DIR, "shared_cov.pkl"))
print("Saved:", SHARED_COV.start_time().date(), "→", SHARED_COV.end_time().date())

Saved: 2023-04-01 → 2026-12-31


In [67]:
DATA_ROOT = os.getcwd()
CACHE_DIR  = os.path.join(DATA_ROOT, "series_cache_scooter")
OUT_DIR    = os.path.join(DATA_ROOT, "predictions_2026")

In [69]:
MODEL_NAME

'daily_tft_festive_lazy__scooters_2026-08-02_16_06_15'

In [70]:
import numpy as np, os

H = 154
tot_last_H = 0.0
tot_365    = 0.0
for k in series_keys:
    with np.load(os.path.join(CACHE_DIR, f"{safe_name(k)}.npz")) as z:
        s = z["val_sales"]          # ends 2026-07-30, day before FORECAST_START
    tot_last_H += s[-H:].sum()
    tot_365    += s[-365:].sum()

print(f"series counted          : {len(series_keys):,}")
print(f"ACTUAL sales, last {H}d : {tot_last_H:,.0f}  ({tot_last_H/1e5:.2f} lacs)")
print(f"ACTUAL sales, last 365d : {tot_365:,.0f}  ({tot_365/1e5:.2f} lacs)")
print()
print(f"model predicted         : 0.10 lacs")
print(f"you expected            : 1.90 lacs")

series counted          : 32,568
ACTUAL sales, last 154d : 173,041  (1.73 lacs)
ACTUAL sales, last 365d : 411,025  (4.11 lacs)

model predicted         : 0.10 lacs
you expected            : 1.90 lacs
